## 1. Install dependencies (run on Colab)

In [ ]:
!pip install -q torch torchvision datasets scikit-learn matplotlib seaborn grad-cam captum

## 2. Clone repo + Set path

In [ ]:
import os
import sys

REPO_DIR = "/content/DL2026-Group5-Project30"
REPO_URL = "https://github.com/USTH-B3-Projects/DL2026-Group5-Project30.git"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

sys.path.append(REPO_DIR)
os.chdir(REPO_DIR)

## 3. Import shared code

In [ ]:
import torch
import torch.nn as nn

from src.datasets.glaucoma_dataset import build_dataloaders
from src.models.cnn_scratch import CNNScratch
from src.training.config import (
    DEVICE, NUM_CLASSES, LEARNING_RATE, WEIGHT_DECAY,
    NUM_EPOCHS, CHECKPOINT_DIR, set_seed
)
from src.training.train_loop import fit, evaluate
from src.utils.metrics import compute_metrics, save_metrics, print_report
from src.utils.visualize import plot_training_curves, plot_confusion_matrix

MODEL_NAME = "cnn_scratch_v2"

## 4. Seed + Load data

In [ ]:
set_seed()

train_loader, val_loader, test_loader = build_dataloaders()
print(f"train bathces: {len(train_loader)} | Val batches: {len(val_loader)} | Test batches: {len(test_loader)}")

### Add weighted loss

In [ ]:
from collections import Counter
from src.datasets.glaucoma_dataset import LABEL_MAP, IDX_TO_CLASS

# Counts samples per class in the (deduped) TRAIN set only
train_classes = train_loader.dataset.data["class"]   # list of "normal"/"early"/"advanced"
counts = Counter(train_classes)
print("Train class counts: ", counts)

num_classes = len(LABEL_MAP)
class_counts = [counts[IDX_TO_CLASS[i]] for i in range(num_classes)]   # order: normal, early, advanced
total = sum(class_counts)

weights = [total / (num_classes * c) for c in class_counts]
class_weights = torch.tensor(weights, dtype=torch.float32).to(DEVICE)
print("Class weights: (normal, early, advanced): ", class_weights.tolist())

## 5. Build model + Optimizer

In [ ]:
model = CNNScratch(num_classes=NUM_CLASSES).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")

loss_fn = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

## 6. Train

In [ ]:
history = fit(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    loss_fn=loss_fn,
    optimizer=optimizer,
    num_epochs=30,
    model_name=MODEL_NAME,             # saves the best checkpoint to checkpoints/cnn_scratch_best.pt
    patience=8,                        # stop early if val_loss doesn't improve for 5 epochs
)

## 7. Plot train/val loss curves

In [ ]:
fig = plot_training_curves(history, model_name=MODEL_NAME)
fig.show()

## 8. Reload the BEST checkpoint before testing

In [ ]:
# The model in memory is from the LAST epoch, which isn't necessarily the
# best one if early stopping kicked in after val_loss had already risen again.

check_ckpt = os.path.join(CHECKPOINT_DIR, f"{MODEL_NAME}_best.pt")
model.load_state_dict(torch.load(check_ckpt, map_location=DEVICE))
print(f"Loaded best checkpoint from {check_ckpt}")

## 9. Evaluate on the test set

In [ ]:
test_loss, test_preds, test_labels = evaluate(model, test_loader, loss_fn)
print(f"Test loss: {test_loss:.4f}")

## 10. Metrics

In [ ]:
metrics = compute_metrics(test_labels, test_preds)

print(f"Accuracy : {metrics['accuracy']:.4f}")
print(f"F1 macro : {metrics['f1_macro']:.4f}")
print()
print_report(test_labels, test_preds)

save_metrics(metrics, model_name=MODEL_NAME)

## 11. Confusion matrix

In [ ]:
fig_cm = plot_confusion_matrix(metrics["confusion_matrix"], model_name=MODEL_NAME)
fig_cm.show()